# Tree-RL: Tree-Structured Reinforcement Learning for Sequential Object Localization
**Reproduce bài báo NIPS 2016** — Zequn Jie et al.

### Thứ tự chạy:
1. Cell 1: Cài packages
2. Cell 2: Download PASCAL VOC
3. Cell 3: Tất cả class/function definitions
4. Cell 4: Extract features (chạy 1 lần)
5. Cell 5: Training
6. Cell 6: Evaluation


In [2]:
# ============================================================
# CELL 1: Cài đặt packages
# ============================================================
import sys
!{sys.executable} -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128 -q
!{sys.executable} -m pip install opencv-python PyYAML tqdm scipy matplotlib tensorboard lxml -q
print('Done!')

Done!


In [3]:
# ============================================================
# CELL 1.5: Mount Google Drive and set project folder
# ============================================================
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/tree_rl_project')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f'Project folder: {PROJECT_DIR}')
except Exception as e:
    # Non-Colab fallback: keep using the current local folder.
    PROJECT_DIR = Path.cwd()
    print(f'Google Drive mount skipped ({type(e).__name__}: {e})')
    print(f'Project folder: {PROJECT_DIR}')

print(f'Current working directory: {Path.cwd()}')


Mounted at /content/drive
Project folder: /content/drive/MyDrive/tree_rl_project
Current working directory: /content/drive/MyDrive/tree_rl_project


In [4]:
# ============================================================
# CELL 2: Download PASCAL VOC 2007 + 2012
# ============================================================
import os, sys, tarfile, urllib.request
from pathlib import Path

URLS = {
    'VOC2007_trainval': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar',
    'VOC2007_test':     'http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar',
    'VOC2012_trainval': 'http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar',
}
DATA_DIR = Path('./VOCdevkit')
DATA_DIR.mkdir(exist_ok=True)
tar_dir = DATA_DIR / '_tars'
tar_dir.mkdir(exist_ok=True)

def download_progress(url, dest):
    fname = url.split('/')[-1]
    fpath = dest / fname
    if fpath.exists():
        print(f'  [Skip] {fname} already exists')
        return fpath
    print(f'  Downloading {fname}...')
    def hook(c, bs, ts):
        if ts > 0:
            sys.stdout.write(f'\r  {c*bs*100//ts}% ({c*bs/1e6:.0f}/{ts/1e6:.0f} MB)')
            sys.stdout.flush()
    urllib.request.urlretrieve(url, fpath, hook)
    print()
    return fpath

for name, url in URLS.items():
    year = '2007' if '2007' in name else '2012'
    voc_dir = DATA_DIR / f'VOC{year}'
    print(f'\n=== {name} ===')
    tar_path = download_progress(url, tar_dir)
    if not voc_dir.exists() or not any(voc_dir.iterdir()):
        print(f'  Extracting...')
        with tarfile.open(tar_path) as tar:
            tar.extractall(DATA_DIR)
        print(f'  Done -> {voc_dir}')
    else:
        print(f'  [Skip] {voc_dir} already extracted')

# Xác định đúng path (tránh lồng VOCdevkit/VOCdevkit)
if (DATA_DIR / 'VOCdevkit' / 'VOC2007').exists():
    VOC_ROOT = str(DATA_DIR / 'VOCdevkit')
else:
    VOC_ROOT = str(DATA_DIR)
print(f'\nVOC_ROOT = {VOC_ROOT}')



=== VOC2007_trainval ===
  100% (460/460 MB)
  Extracting...


/tmp/ipykernel_4849/3920232345.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATA_DIR)


  Done -> VOCdevkit/VOC2007

=== VOC2007_test ===
  100% (451/451 MB)
  Extracting...
  Done -> VOCdevkit/VOC2007

=== VOC2012_trainval ===
  100% (2000/2000 MB)
  Extracting...
  Done -> VOCdevkit/VOC2012

VOC_ROOT = VOCdevkit/VOCdevkit


In [5]:
# ============================================================
# CELL 3: Tất cả definitions (Actions, Metrics, Memory,
#          Dataset, FeatureExtractor, QNetwork, Agent)
# ============================================================
import random, copy, os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import deque, namedtuple
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.ops as ops
import torchvision.transforms as T
from torch.utils.data import Dataset
from PIL import Image
from tqdm.notebook import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name} ({total_gb:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

# ── CONFIG: fast A100 experimental preset ────────────────────
# This version keeps a safer speed/accuracy tradeoff: Double DQN for
# stability, paper-style 50-step episodes, and fewer optimizer updates.
# Use tree_rl_paper_repro.ipynb for strict paper-faithful runs.
CFG = {
    'voc_root':          VOC_ROOT,
    'feature_cache_dir': './feature_cache',
    'checkpoint_dir':    './checkpoints_ddqn',
    'log_dir':           './runs/tree_rl_ddqn',
    # Model
    'roi_pool_size':     7,
    'hidden_dim':        2048,
    'num_actions':       13,
    'history_len':       50,
    # Actions
    'scaling_ratio':     0.55,
    'trans_ratio':       0.25,
    # Reward
    'iou_threshold':     0.5,
    'first_hit_bonus':   5.0,
    # Training
    'epochs':            25,
    'max_steps':         50,
    'batch_size':        1024,
    'lr':                3e-4,
    'weight_decay':      1e-5,
    'gamma':             0.9,
    'eps_start':         1.0,
    'eps_end':           0.05,
    'eps_decay_epochs':  10,
    'replay_capacity':   300_000,
    'replay_start':      20_000,
    'target_update_freq':1000,
    'update_every':      2,
    'double_dqn':        True,
    'resume':            True,
    'save_every':        1,
    'grad_clip':         10.0,
    'use_amp':           True,
    # State dim = 4096 + 4096 + 50*13
    'state_dim':         8842,
}
Path(CFG['checkpoint_dir']).mkdir(exist_ok=True)
Path(CFG['feature_cache_dir']).mkdir(exist_ok=True)
Path(CFG['log_dir']).mkdir(parents=True, exist_ok=True)

# ── ACTIONS ──────────────────────────────────────────────────
ACTION_NAMES = [
    'scale_top_left','scale_top_right','scale_bottom_left',
    'scale_bottom_right','scale_center',
    'trans_left','trans_right','trans_up','trans_down',
    'trans_wider','trans_narrower','trans_taller','trans_shorter',
]

def apply_action(window, action_id, sr=0.55, tr=0.25, img_w=1, img_h=1):
    x1,y1,x2,y2 = window
    w,h = x2-x1, y2-y1
    if action_id < 5:  # scaling
        sw,sh = w*sr, h*sr
        offsets = [(0,0),(w-sw,0),(0,h-sh),(w-sw,h-sh),((w-sw)/2,(h-sh)/2)]
        dx,dy = offsets[action_id]
        nx1,ny1,nx2,ny2 = x1+dx, y1+dy, x1+dx+sw, y1+dy+sh
    else:  # translation
        dx,dy = w*tr, h*tr
        moves = [(-dx,0),(dx,0),(0,-dy),(0,dy),(-dx,0,dx,0),(dx,0,-dx,0),(0,-dy,0,dy),(0,dy,0,-dy)]
        aid = action_id - 5
        if aid == 0:   nx1,ny1,nx2,ny2 = x1-dx,y1,x2-dx,y2
        elif aid == 1: nx1,ny1,nx2,ny2 = x1+dx,y1,x2+dx,y2
        elif aid == 2: nx1,ny1,nx2,ny2 = x1,y1-dy,x2,y2-dy
        elif aid == 3: nx1,ny1,nx2,ny2 = x1,y1+dy,x2,y2+dy
        elif aid == 4: nx1,ny1,nx2,ny2 = x1-dx,y1,x2+dx,y2
        elif aid == 5: nx1,ny1,nx2,ny2 = x1+dx,y1,x2-dx,y2
        elif aid == 6: nx1,ny1,nx2,ny2 = x1,y1-dy,x2,y2+dy
        else:          nx1,ny1,nx2,ny2 = x1,y1+dy,x2,y2-dy
    nx1 = max(0, min(nx1, img_w-2))
    ny1 = max(0, min(ny1, img_h-2))
    nx2 = max(nx1+1, min(nx2, img_w))
    ny2 = max(ny1+1, min(ny2, img_h))
    return np.array([nx1,ny1,nx2,ny2], dtype=np.float32)

def encode_history(history, hlen=50, nact=13):
    enc = np.zeros(hlen*nact, dtype=np.float32)
    for i,a in enumerate(history[-hlen:]):
        enc[i*nact+a] = 1.0
    return enc

# ── METRICS ──────────────────────────────────────────────────
def iou(a, b):
    ix1,iy1 = max(a[0],b[0]), max(a[1],b[1])
    ix2,iy2 = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union>0 else 0.0

def compute_reward(cur, nxt, gts, hit_flags, thr=0.5, bonus=5.0):
    new_flags = hit_flags.copy()
    for i,gt in enumerate(gts):
        if hit_flags[i]<1 and iou(nxt,gt)>=thr:
            new_flags[i]=1
            return bonus, new_flags
    max_sign = -1.0
    for gt in gts:
        if iou(nxt,gt)-iou(cur,gt)>0:
            max_sign=1.0; break
    return max_sign, new_flags

def recall_curve(proposals_list, gt_list, iou_thrs=[0.5,0.6,0.7,0.8], size_thr=2000):
    res = {'all':{},'large':{},'small':{}}
    for thr in iou_thrs:
        counts = {'all':[0,0],'large':[0,0],'small':[0,0]}
        for props,gts in zip(proposals_list,gt_list):
            for gt in gts:
                area = (gt[2]-gt[0])*(gt[3]-gt[1])
                sk = 'large' if area>size_thr else 'small'
                hit = int(any(iou(p,gt)>=thr for p in props))
                counts['all'][0]+=hit; counts['all'][1]+=1
                counts[sk][0]+=hit;   counts[sk][1]+=1
        for k in res:
            tot = counts[k][1]
            res[k][thr] = counts[k][0]/tot if tot>0 else 0.0
    return res

# ── REPLAY MEMORY (preallocated for large A100 batches) ──────
class ReplayMemory:
    def __init__(self, capacity=500_000, state_dim=8842):
        self.capacity = int(capacity)
        self.state_dim = int(state_dim)
        self.pos = 0
        self.size = 0
        self.states = np.empty((self.capacity, self.state_dim), dtype=np.float16)
        self.next_states = np.empty((self.capacity, self.state_dim), dtype=np.float16)
        self.actions = np.empty(self.capacity, dtype=np.int16)
        self.rewards = np.empty(self.capacity, dtype=np.float32)
        self.dones = np.empty(self.capacity, dtype=np.float32)

    def push(self, s, a, r, s_, done):
        self.states[self.pos] = s.astype(np.float16, copy=False)
        self.next_states[self.pos] = s_.astype(np.float16, copy=False)
        self.actions[self.pos] = int(a)
        self.rewards[self.pos] = float(r)
        self.dones[self.pos] = float(done)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, bs=1024):
        idx = np.random.randint(0, self.size, size=bs)
        return (
            self.states[idx].astype(np.float32),
            self.actions[idx].astype(np.int64),
            self.rewards[idx],
            self.next_states[idx].astype(np.float32),
            self.dones[idx],
        )

    def __len__(self):
        return self.size

# ── DATASET ──────────────────────────────────────────────────
VOC_CLASSES = [
    'aeroplane','bicycle','bird','boat','bottle','bus','car','cat',
    'chair','cow','diningtable','dog','horse','motorbike','person',
    'pottedplant','sheep','sofa','train','tvmonitor'
]
CLS2IDX = {c:i for i,c in enumerate(VOC_CLASSES)}

class VOCDataset(Dataset):
    def __init__(self, root, years=['2007','2012'], split='trainval'):
        self.root=root; self.split=split
        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
        self.samples=[]
        for year in years:
            sf=os.path.join(root,f'VOC{year}','ImageSets','Main',f'{split}.txt')
            with open(sf) as f:
                self.samples+=[(year,l.strip()) for l in f if l.strip()]
        print(f'[VOCDataset] {split}: {len(self.samples)} images from {years}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        year,iid = self.samples[idx]
        img = Image.open(os.path.join(self.root,f'VOC{year}','JPEGImages',f'{iid}.jpg')).convert('RGB')
        W,H = img.size
        boxes,labels=[],[]
        ann = os.path.join(self.root,f'VOC{year}','Annotations',f'{iid}.xml')
        for obj in ET.parse(ann).getroot().findall('object'):
            diff=obj.find('difficult')
            if diff is not None and int(diff.text)==1: continue
            cls=obj.find('name').text.strip().lower()
            if cls not in CLS2IDX: continue
            bb=obj.find('bndbox')
            x1=max(0,float(bb.find('xmin').text)-1)
            y1=max(0,float(bb.find('ymin').text)-1)
            x2=min(W,float(bb.find('xmax').text)-1)
            y2=min(H,float(bb.find('ymax').text)-1)
            if x2>x1 and y2>y1:
                boxes.append([x1,y1,x2,y2]); labels.append(CLS2IDX[cls])
        gt = np.array(boxes,dtype=np.float32) if boxes else np.zeros((0,4),dtype=np.float32)
        return {'image':self.transform(img),'img_id':iid,'year':year,
                'img_w':W,'img_h':H,'gt_boxes':gt}

# ── FEATURE EXTRACTOR ────────────────────────────────────────
class VGG16Extractor(nn.Module):
    def __init__(self, roi_size=7):
        super().__init__()
        self.roi_size = roi_size
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.conv = nn.Sequential(*list(vgg.features.children())[:30])
        self.fc6  = nn.Sequential(
            nn.Linear(512*roi_size*roi_size, 4096),
            nn.ReLU(inplace=True), nn.Dropout(0.5)
        )
        if roi_size==7:
            self.fc6[0].weight.data = vgg.classifier[0].weight.data
            self.fc6[0].bias.data   = vgg.classifier[0].bias.data
        for p in self.conv.parameters(): p.requires_grad=False

    def get_fmap(self, img):
        with torch.no_grad(): return self.conv(img)

    def roi_feat(self, fmap, boxes, scale=1/16):
        bidx = torch.zeros(len(boxes),1,dtype=boxes.dtype,device=boxes.device)
        rois = torch.cat([bidx,boxes],1)
        pooled = ops.roi_pool(fmap,rois,(self.roi_size,self.roi_size),scale)
        return self.fc6(pooled.view(len(boxes),-1))

# ── Q-NETWORK ────────────────────────────────────────────────
class QNetwork(nn.Module):
    def __init__(self, state_dim=8842, hidden=1024, n_act=13):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim,hidden), nn.ReLU(),
            nn.Linear(hidden,hidden),   nn.ReLU(),
            nn.Linear(hidden,hidden),   nn.ReLU(),
            nn.Linear(hidden,n_act),
        )
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias,0)
    def forward(self,x): return self.net(x)
    def two_best(self, s):
        with torch.no_grad():
            q = self.forward(s.unsqueeze(0)).squeeze(0)
            return int(q[:5].argmax()), int(q[5:].argmax())+5
    def greedy(self, s):
        with torch.no_grad():
            return int(self.forward(s.unsqueeze(0)).squeeze(0).argmax())

# ── AGENT ────────────────────────────────────────────────────
class TreeRLAgent:
    def __init__(self, extractor, q_net, target_net, cfg, device):
        self.ext=extractor; self.q=q_net; self.tgt=target_net
        self.dev=device; self.cfg=cfg
        self.roi_scale = 1/16

    def _whole_feat_np(self, fmap, W, H):
        box = torch.tensor([[0,0,W,H]], dtype=torch.float32, device=self.dev)
        with torch.inference_mode():
            return self.ext.roi_feat(fmap, box, self.roi_scale).squeeze(0).cpu().numpy()

    def _window_feat_np(self, fmap, box):
        wb = torch.from_numpy(np.asarray([box], dtype=np.float32)).to(self.dev)
        with torch.inference_mode():
            return self.ext.roi_feat(fmap, wb, self.roi_scale).squeeze(0).cpu().numpy()

    def _build_state_np(self, gf_np, rf_np, hist):
        return np.concatenate([
            rf_np, gf_np,
            encode_history(hist, self.cfg['history_len'], self.cfg['num_actions'])
        ]).astype(np.float32, copy=False)

    def run_episode(self, fmap, W, H, gt_boxes, eps):
        win = np.array([0,0,W,H],dtype=np.float32)
        hist=[]; hit_flags=np.full(len(gt_boxes),-1.0)
        gf_np = self._whole_feat_np(fmap,W,H)
        rf_np = self._window_feat_np(fmap,win)
        transitions=[]
        for step in range(self.cfg['max_steps']):
            state = self._build_state_np(gf_np,rf_np,hist)
            # eps-greedy (tree variant)
            if random.random()<eps:
                act = random.randint(0,12)
            else:
                st = torch.from_numpy(state).to(self.dev)
                bs,bt = self.q.two_best(st)
                act = random.choice([bs,bt])
            nwin = apply_action(win,act,
                self.cfg['scaling_ratio'],self.cfg['trans_ratio'],W,H)
            rew,hit_flags = compute_reward(
                win,nwin,gt_boxes,hit_flags,
                self.cfg['iou_threshold'],self.cfg['first_hit_bonus'])
            hist.append(act)
            nrf_np = self._window_feat_np(fmap,nwin)
            nstate = self._build_state_np(gf_np,nrf_np,hist)
            done = (step==self.cfg['max_steps']-1)
            transitions.append((state,act,rew,nstate,float(done)))
            win=nwin; rf_np=nrf_np
        return transitions

    def tree_search(self, fmap, W, H, levels=5):
        gf_np = self._whole_feat_np(fmap,W,H)
        props=[]; queue=deque()
        queue.append((np.array([0,0,W,H],dtype=np.float32),[],1))
        while queue:
            win,hist,lvl=queue.popleft()
            props.append(win.copy())
            if lvl>=levels: continue
            rf_np = self._window_feat_np(fmap,win)
            state=self._build_state_np(gf_np,rf_np,hist)
            st=torch.from_numpy(state).to(self.dev)
            bs,bt=self.q.two_best(st)
            for a in [bs,bt]:
                nw=apply_action(win,a,self.cfg['scaling_ratio'],
                                self.cfg['trans_ratio'],W,H)
                queue.append((nw,hist+[a],lvl+1))
        return props

print('All definitions loaded OK!')


Device: cuda
GPU: NVIDIA A100-SXM4-40GB (39.5 GB)
All definitions loaded OK!


In [ ]:
# ============================================================
# CELL 4: Pre-compute conv5_3 feature maps (chạy 1 lần)
# Nếu đã có cache rồi thì skip cell này
# ============================================================
extractor = VGG16Extractor(roi_size=CFG['roi_pool_size']).to(DEVICE)
extractor.eval()

transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

def extract_split(root, year, split, fallback_to_cpu=True):
    out_dir = Path(CFG['feature_cache_dir']) / f'VOC{year}' / split
    out_dir.mkdir(parents=True, exist_ok=True)
    sf = os.path.join(root, f'VOC{year}', 'ImageSets', 'Main', f'{split}.txt')
    with open(sf) as f:
        ids = [l.strip() for l in f if l.strip()]

    skipped = 0
    cpu_extractor = None
    for iid in tqdm(ids, desc=f'VOC{year} {split}'):
        out = out_dir / f'{iid}.pt'
        if out.exists():
            skipped += 1
            continue

        img_path = os.path.join(root, f'VOC{year}', 'JPEGImages', f'{iid}.jpg')
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        img_t = transform(img).unsqueeze(0).contiguous().to(DEVICE, non_blocking=True)

        try:
            with torch.inference_mode():
                fmap = extractor.get_fmap(img_t).squeeze(0).contiguous().half().cpu()
        except RuntimeError as e:
            msg = str(e).lower()
            if not fallback_to_cpu or ('cuda' not in msg and 'accelerator' not in msg and 'misaligned' not in msg):
                raise
            print(f'\n[CUDA fallback] {iid}: {e}')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if cpu_extractor is None:
                cpu_extractor = VGG16Extractor(roi_size=CFG['roi_pool_size']).cpu().eval()
            with torch.inference_mode():
                fmap = cpu_extractor.get_fmap(transform(img).unsqueeze(0).contiguous()).squeeze(0).contiguous().half()

        torch.save({'feature_map': fmap, 'img_w': W, 'img_h': H}, out)
        del img_t, fmap
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f'Done: {len(ids)-skipped} extracted, {skipped} skipped')

extract_split(CFG['voc_root'], '2007', 'trainval')
extract_split(CFG['voc_root'], '2012', 'trainval')
extract_split(CFG['voc_root'], '2007', 'test')


VOC2007 trainval:   0%|          | 0/5011 [00:00<?, ?it/s]

Done: 0 extracted, 5011 skipped


VOC2012 trainval:   0%|          | 0/11540 [00:00<?, ?it/s]

Done: 3083 extracted, 8457 skipped


VOC2007 test:   0%|          | 0/4952 [00:00<?, ?it/s]

Done: 4952 extracted, 0 skipped


In [ ]:
# ============================================================
# CELL 5: TRAINING
# ============================================================
import warnings; warnings.filterwarnings('ignore')

def load_fmap(year, split, iid):
    p = Path(CFG['feature_cache_dir'])/f'VOC{year}'/split/f'{iid}.pt'
    if not p.exists(): return None,None,None
    d = torch.load(p, map_location='cpu')
    fmap = d['feature_map'].float().unsqueeze(0).to(DEVICE, non_blocking=True)
    return fmap, d['img_w'], d['img_h']

def get_eps(epoch):
    s,e,d = CFG['eps_start'],CFG['eps_end'],CFG['eps_decay_epochs']
    return e if epoch>=d else s-(s-e)*(epoch/d)

def compute_loss(batch, q, tgt, gamma, double_dqn=True):
    S,A,R,S_,D = batch
    S  = torch.from_numpy(S).to(DEVICE, non_blocking=True)
    A  = torch.from_numpy(A).to(DEVICE, non_blocking=True)
    R  = torch.from_numpy(R).to(DEVICE, non_blocking=True)
    S_ = torch.from_numpy(S_).to(DEVICE, non_blocking=True)
    D  = torch.from_numpy(D).to(DEVICE, non_blocking=True)
    qsa = q(S).gather(1,A.unsqueeze(1)).squeeze(1)
    with torch.no_grad():
        if double_dqn:
            next_actions = q(S_).argmax(1, keepdim=True)
            qnext = tgt(S_).gather(1, next_actions).squeeze(1)
        else:
            qnext = tgt(S_).max(1)[0]
        target = R + gamma*qnext*(1-D)
    return F.smooth_l1_loss(qsa, target)

# Init models
extractor = VGG16Extractor(CFG['roi_pool_size']).to(DEVICE)
extractor.eval()

q_net     = QNetwork(CFG['state_dim'], CFG['hidden_dim'], CFG['num_actions']).to(DEVICE)
target_net= copy.deepcopy(q_net)
target_net.eval()
for p in target_net.parameters(): p.requires_grad=False

agent  = TreeRLAgent(extractor, q_net, target_net, CFG, DEVICE)
memory = ReplayMemory(CFG['replay_capacity'], CFG['state_dim'])
try:
    optimizer = optim.AdamW(
        q_net.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'],
        fused=torch.cuda.is_available()
    )
except TypeError:
    optimizer = optim.AdamW(q_net.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available() and CFG['use_amp'])

# Dataset
train_ds = VOCDataset(CFG['voc_root'], years=['2007','2012'], split='trainval')

# Resume nếu có checkpoint
start_epoch=0; global_step=0
ckpt_dir=Path(CFG['checkpoint_dir'])
ckpt_dir.mkdir(parents=True, exist_ok=True)
existing = sorted(ckpt_dir.glob('epoch_*.pth'))
if CFG.get('resume', True) and existing:
    ckpt_path = existing[-1]
    ckpt=torch.load(ckpt_path, map_location=DEVICE)
    q_net.load_state_dict(ckpt['q_net'])
    target_net.load_state_dict(ckpt['target_net'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if 'scaler' in ckpt:
        scaler.load_state_dict(ckpt['scaler'])
    start_epoch=ckpt['epoch']+1
    global_step=ckpt.get('global_step',0)
    if 'rng' in ckpt:
        random.setstate(ckpt['rng']['python'])
        np.random.set_state(ckpt['rng']['numpy'])
        torch.set_rng_state(ckpt['rng']['torch_cpu'])
        if torch.cuda.is_available() and ckpt['rng'].get('torch_cuda') is not None:
            torch.cuda.set_rng_state_all(ckpt['rng']['torch_cuda'])
    print(f'Resumed from {ckpt_path} (next epoch {start_epoch+1}/{CFG["epochs"]}, global_step={global_step})')
else:
    print(f'Starting fresh. Checkpoints will be saved to: {ckpt_dir.resolve()}')

print(f'Training Tree-RL | {len(train_ds)} images | {CFG["epochs"]} epochs')
print(f'Fast A100 | DDQN: {CFG["double_dqn"]} | Steps: {CFG["max_steps"]} | Update every: {CFG["update_every"]}')
print(f'Hidden: {CFG["hidden_dim"]} | Replay: {CFG["replay_capacity"]:,} | Batch: {CFG["batch_size"]} | GPU: {DEVICE}')
print('='*60)

for epoch in range(start_epoch, CFG['epochs']):
    eps = get_eps(epoch)
    indices = list(range(len(train_ds)))
    random.shuffle(indices)
    losses,rewards=[],[]
    q_net.train()

    pbar = tqdm(indices, desc=f'Epoch {epoch+1}/{CFG["epochs"]} ε={eps:.2f}')
    for idx in pbar:
        s=train_ds[idx]
        if len(s['gt_boxes'])==0: continue

        fmap,W,H = load_fmap(s['year'],'trainval',s['img_id'])
        if fmap is None:
            img_t=s['image'].unsqueeze(0).to(DEVICE)
            with torch.no_grad(): fmap=extractor.get_fmap(img_t)
            W,H=s['img_w'],s['img_h']

        trans=agent.run_episode(fmap,W,H,s['gt_boxes'],eps)
        ep_r=0
        for (st,a,r,st_,d) in trans:
            memory.push(st,a,r,st_,d); ep_r+=r
        rewards.append(ep_r)

        if (len(memory)>=CFG['replay_start'] and
            len(memory)>=CFG['batch_size'] and
            len(rewards)%CFG['update_every']==0):
            batch=memory.sample(CFG['batch_size'])
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available() and CFG['use_amp']):
                loss=compute_loss(batch,q_net,target_net,CFG['gamma'],CFG['double_dqn'])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(q_net.parameters(), CFG['grad_clip'])
            scaler.step(optimizer); scaler.update()
            losses.append(loss.item())
            global_step+=1
            if global_step%CFG['target_update_freq']==0:
                target_net.load_state_dict(q_net.state_dict())

        if len(losses)>0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.4f}',
                             reward=f'{np.mean(rewards[-50:]):.1f}',
                             mem=len(memory))

    al=np.mean(losses) if losses else 0
    ar=np.mean(rewards) if rewards else 0
    print(f'Epoch {epoch+1}: loss={al:.4f}, reward={ar:.2f}, mem={len(memory):,}')

    if (epoch + 1) % CFG['save_every'] == 0:
        ckpt_path = ckpt_dir/f'epoch_{epoch+1:03d}.pth'
        tmp_path = ckpt_dir/f'epoch_{epoch+1:03d}.tmp.pth'
        rng = {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch_cpu': torch.get_rng_state(),
            'torch_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        torch.save({'epoch':epoch,'global_step':global_step,
                    'q_net':q_net.state_dict(),'target_net':target_net.state_dict(),
                    'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
                    'cfg':CFG.copy(),'rng':rng}, tmp_path)
        tmp_path.replace(ckpt_path)
        print(f'Saved checkpoint: {ckpt_path}')

print('Training complete!')


[VOCDataset] trainval: 16551 images from ['2007', '2012']
Starting fresh. Checkpoints will be saved to: /content/drive/MyDrive/tree_rl_project/checkpoints_ddqn
Training Tree-RL | 16551 images | 25 epochs
Fast A100 | DDQN: True | Steps: 50 | Update every: 2
Hidden: 2048 | Replay: 300,000 | Batch: 1024 | GPU: cuda


Epoch 1/25 ε=1.00:   0%|          | 0/16551 [00:00<?, ?it/s]

Epoch 1: loss=0.0853, reward=-34.55, mem=300,000
Saved checkpoint: checkpoints_ddqn/epoch_001.pth


Epoch 2/25 ε=0.91:   0%|          | 0/16551 [00:00<?, ?it/s]

Epoch 2: loss=0.1090, reward=-31.65, mem=300,000
Saved checkpoint: checkpoints_ddqn/epoch_002.pth


...

Epoch 23: loss=0.0844, reward=-2.84, mem=300,000
Saved checkpoint: checkpoints_ddqn/epoch_023.pth


Epoch 24/25 ε=0.05:   0%|          | 0/16551 [00:00<?, ?it/s]

Epoch 24: loss=0.0855, reward=-2.95, mem=300,000
Saved checkpoint: checkpoints_ddqn/epoch_024.pth


Epoch 25/25 ε=0.05:   0%|          | 0/16551 [00:00<?, ?it/s]

: 

In [10]:
# ============================================================
# CELL 6: EVALUATION — Recall table (Table 2 bài báo)
# ============================================================

# Load checkpoint tốt nhất
ckpt_dir = Path(CFG['checkpoint_dir'])
existing = sorted(ckpt_dir.glob('epoch_*.pth'))
if not existing:
    raise FileNotFoundError('Chưa có checkpoint! Chạy Cell 5 trước.')

ckpt = torch.load(existing[-1], map_location=DEVICE, weights_only=False)
q_net.load_state_dict(ckpt['q_net'])
q_net.eval()
print(f'Loaded: {existing[-1].name}')

test_ds = VOCDataset(CFG['voc_root'], years=['2007'], split='test')

def run_eval(levels, max_images=None):
    all_props, all_gts = [], []
    n = len(test_ds) if max_images is None else min(max_images, len(test_ds))
    for i in tqdm(range(n), desc=f'Tree search levels={levels}'):
        s=test_ds[i]
        if len(s['gt_boxes'])==0: continue
        fmap,W,H=load_fmap(s['year'],'test',s['img_id'])
        if fmap is None:
            img_t=s['image'].unsqueeze(0).to(DEVICE)
            with torch.no_grad(): fmap=extractor.get_fmap(img_t)
            W,H=s['img_w'],s['img_h']
        with torch.no_grad():
            props=agent.tree_search(fmap,W,H,levels=levels)
        all_props.append(props)
        all_gts.append(s['gt_boxes'])

    res=recall_curve(all_props,all_gts)
    n_props=2**levels-1
    print(f'\n{"="*60}')
    print(f'Tree-RL Recall | Levels={levels}, Proposals={n_props}')
    print(f'{"="*60}')
    print(f'{"Category":<10} {"IoU=0.5":>9} {"IoU=0.6":>9} {"IoU=0.7":>9} {"IoU=0.8":>9}')
    print(f'{"-"*50}')

    # Paper reference values for levels=5 (31 proposals)
    paper = {'all':{0.5:68.1,0.6:58.7,0.7:43.8},
             'large':{0.5:78.9,0.6:69.8,0.7:53.3},
             'small':{0.5:23.2,0.6:12.5,0.7:4.5}}

    for k in ['all','large','small']:
        vals=[res[k].get(t,0)*100 for t in [0.5,0.6,0.7,0.8]]
        row=f'{k.capitalize():<10} '+' '.join(f'{v:>9.1f}' for v in vals)
        # Thêm so sánh với paper nếu levels=5
        if levels==5:
            refs=[paper[k].get(t,0) for t in [0.5,0.6,0.7]]
            diffs=[f'({v-r:+.1f})' for v,r in zip(vals[:3],refs)]
            row+='  vs paper: '+' '.join(diffs)
        print(row)
    print(f'{"="*60}')
    return res

# Chạy với 31 proposals (5 levels) và 63 proposals (6 levels)
res_31 = run_eval(levels=5)   # 31 proposals — Table 2 bài báo
res_63 = run_eval(levels=6)   # 63 proposals

Loaded: epoch_025.pth
[VOCDataset] test: 4952 images from ['2007']


Tree search levels=5:   0%|          | 0/4952 [00:00<?, ?it/s]


Tree-RL Recall | Levels=5, Proposals=31
Category     IoU=0.5   IoU=0.6   IoU=0.7   IoU=0.8
--------------------------------------------------
All             42.3      31.0      18.8       8.4  vs paper: (-25.8) (-27.7) (-25.0)
Large           47.5      34.9      21.2       9.4  vs paper: (-31.4) (-34.9) (-32.1)
Small            1.1       0.2       0.0       0.0  vs paper: (-22.1) (-12.3) (-4.5)


Tree search levels=6:   0%|          | 0/4952 [00:00<?, ?it/s]


Tree-RL Recall | Levels=6, Proposals=63
Category     IoU=0.5   IoU=0.6   IoU=0.7   IoU=0.8
--------------------------------------------------
All             47.6      35.4      21.7       9.7
Large           53.2      39.8      24.5      11.0
Small            3.3       1.1       0.3       0.1
